# XFND-CLIP — Explainable Multimodal Fake News Detection (IFND)
Run the cells **top to bottom**. Before starting: `Runtime → Change runtime type → T4 GPU`.

Rough timings on a free T4: setup 3 min · image download 5–15 min · feature extraction 5–10 min · training 5–10 min · evaluation/explanations 3 min.

Everything you need for the report ends up in `results/` and is zipped at the end.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("CUDA available:", torch.cuda.is_available())

## 2. Get the project code
Upload `XFND-CLIP.zip` (or the extracted XFND-CLIP folder) anywhere in your Google Drive. If it is not found, the cell asks you to upload the zip directly.

In [ ]:
from google.colab import drive, files
import glob, os, shutil, zipfile
drive.mount('/content/drive')
os.chdir('/content')
shutil.rmtree('/content/XFND-CLIP', ignore_errors=True)

# 1) an already-extracted project folder anywhere in Drive
folders = [os.path.dirname(p) for p in glob.glob('/content/drive/MyDrive/**/XFND-CLIP*/config.py', recursive=True)]
# 2) the zip anywhere in Drive (also catches "XFND-CLIP (1).zip")
zips = glob.glob('/content/drive/MyDrive/**/XFND-CLIP*.zip', recursive=True)
zips = [z for z in zips if 'results' not in z.lower()]

if folders:
    print('Using folder:', folders[0]); shutil.copytree(folders[0], '/content/XFND-CLIP')
else:
    if zips:
        src = zips[0]; print('Using zip:', src)
    else:
        print('Project zip not found in Drive - please upload XFND-CLIP.zip now')
        src = list(files.upload())[0]
    with zipfile.ZipFile(src) as z:
        z.extractall('/content/_xfnd')
    cfg = glob.glob('/content/_xfnd/**/config.py', recursive=True)[0]
    shutil.move(os.path.dirname(cfg), '/content/XFND-CLIP'); shutil.rmtree('/content/_xfnd', ignore_errors=True)

os.chdir('/content/XFND-CLIP')
print(os.getcwd()); print(sorted(os.listdir()))

## 3. Install packages

In [ ]:
!pip install -q -r requirements.txt

## 4. Put the IFND dataset in `data/`
Download the IFND csv (Sharma & Garg, *Complex & Intelligent Systems*, 2021 — search Kaggle for **"IFND Indian Fake News Dataset"**).
Then **either** upload it with the cell below **or** copy it from your Drive.
The file must end up as `data/IFND.csv`.

In [ ]:
import os, shutil
os.makedirs('data', exist_ok=True)
from google.colab import files
if os.path.exists('/content/drive/MyDrive/IFND.csv'):
    shutil.copy('/content/drive/MyDrive/IFND.csv', 'data/IFND.csv')
else:
    up = files.upload()                      # pick the IFND csv file
    name = list(up)[0]
    shutil.move(name, 'data/IFND.csv')
import pandas as pd
for enc in ('utf-8', 'latin-1'):
    try:
        raw = pd.read_csv('data/IFND.csv', encoding=enc); break
    except UnicodeDecodeError: pass
print(raw.shape); print(raw.columns.tolist()); raw.head()

## 5. Clean the data, download images, make splits
This keeps every fake item and samples real items up to 8,000 per class.
**Look at `image_coverage_pct` in the output.** Many old IFND image links are dead; if coverage is below ~20 %, tell me and we can switch to Fakeddit.

In [ ]:
!python data_prep.py

## 6. Extract features with the frozen encoders (BERT, ResNet-50, CLIP)

In [ ]:
!python features.py

## 7. Train all models (baselines, FND-CLIP, XFND-CLIP, ablations — 3 seeds each)

In [ ]:
!python train.py

## 8. Tables and figures

In [ ]:
!python evaluate.py
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('results/figures/*.png')):
    print(f); display(Image(f, width=650))

## 9. Explanations for test-set examples

In [ ]:
!python explain.py --from_test 6
for f in sorted(glob.glob('results/figures/explanations/*.png')):
    print(f); display(Image(f, width=800))

## 10. Real Indian examples (your own live test)
Pick 4–6 recent posts that **PIB Fact Check** or **Alt News / BOOM / Factly** have already checked — some fake, some genuine.
Save the image from each post, upload them below, and type the headline and the fact-checker's verdict.
These become the "real-world testing" section of the report. Use only posts that were published *after* 2021 so they cannot be in IFND.

In [ ]:
from google.colab import files
os.makedirs('live_test', exist_ok=True)
up = files.upload()                    # upload the images you saved
for n in up: shutil.move(n, f'live_test/{n}')
print(os.listdir('live_test'))

In [ ]:
# Fill this list yourself: (image file name or None, headline text, fact-checker verdict, source link)
LIVE = [
    # ("post1.jpg", "Headline exactly as it was shared ...", "FAKE", "https://..."),
    # ("post2.jpg", "...", "REAL", "https://..."),
]
from explain import Explainer
from encoders import load_image
ex = Explainer()
rows = []
os.makedirs('results/figures/live', exist_ok=True)
for i, (img_name, text, truth, link) in enumerate(LIVE, 1):
    img = load_image(f'live_test/{img_name}') if img_name else None
    p = ex.figure(text, img, f'results/figures/live/live_{i}.png', 1 if truth == 'FAKE' else 0)
    rows.append({'#': i, 'headline': text, 'fact_check': truth, 'model': p['label'],
                 'P(fake)': round(p['prob_fake'], 3), 'clip_sim': p['clip_sim'], 'source': link})
    display(Image(f'results/figures/live/live_{i}.png', width=800))
live_df = pd.DataFrame(rows); live_df.to_csv('results/live_test.csv', index=False); live_df

## 11. Web demo
Run the cell, open the **public URL** it prints, try a few headlines and take screenshots for the report/PPT.
Stop the cell (■) when you are done. Every query is logged in `results/demo_log.csv`.

In [ ]:
!python app.py --share

## 12. Package the results
This creates `XFND_results.zip` in your Drive. **Send me that zip** (or `metrics.json`, `data_stats.json`, the csv tables and the `figures` folder) and I will write the Results chapter and results slides from your numbers.

In [ ]:
!cp data/processed.csv results/ 2>/dev/null; nvidia-smi --query-gpu=name --format=csv,noheader > results/gpu.txt
!cd /content/XFND-CLIP && zip -qr /content/drive/MyDrive/XFND_results.zip results checkpoints/xfnd_clip.pt
!ls -lh /content/drive/MyDrive/XFND_results.zip